In [ ]:
# ── CELDA 1: Setup y datos (versión web — kit precalculado) ──────────────────
import os, json, warnings
warnings.filterwarnings('ignore')
from functools import lru_cache
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, HTML
from matplotlib.lines import Line2D

try:
    from scipy.ndimage import gaussian_filter
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False
_HAS_SPH = False   # py-sphviewer no corre en el navegador (WebAssembly)

# ── Kit de datos precalculado (generado por make_kit_data.py) ────────────────
from pathlib import Path
for DATA in ('data', '/files/data', '/drive/data', 'files/data', '/files', '.'):
    if Path(DATA, 'meta.json').exists():
        break
else:
    raise FileNotFoundError('no encuentro data/meta.json en ninguna ruta conocida')
with open(f'{DATA}/meta.json') as f:
    META = json.load(f)

sim_name    = META['sim']
gx_id       = META['gx']
_T_UNIVERSE = META['t_universe']
snap_cache  = {int(s): (z, t)
               for s, z, t in zip(META['snaps'], META['z'], META['t_lb'])}
_snaps_by_time = sorted(snap_cache, key=lambda s: snap_cache[s][1], reverse=True)

def _arr(vals):
    return np.array([v if v is not None else np.nan for v in vals], dtype=float)

HIST_T    = _arr(META['hist']['t_age'])
HIST_MS   = _arr(META['hist']['mstar'])
HIST_MG   = _arr(META['hist']['mgas'])
HIST_LSFR = _arr(META['hist']['lsfr'])

# ── Componentes: todas las estrellas y todo el gas ───────────────────────────
ALL_KEYS = ['gas', 'stars']
NOMBRES  = {'stars': 'Estrellas', 'gas': 'Gas'}
_EMPTY   = (np.array([]), np.array([]))

@lru_cache(maxsize=32)
def load_epoch(snap, coord_type):
    """Devuelve {'stars': (x, y), 'gas': (x, y)} desde el kit .npz."""
    path = f'{DATA}/{snap:03d}_{coord_type}.npz'
    comps = {}
    if os.path.exists(path):
        d = np.load(path)
        if 'sx' in d:
            comps['stars'] = (d['sx'], d['sy'])
        if 'gx' in d:
            comps['gas'] = (d['gx'], d['gy'])
    return comps

def _fmt_n(n):
    return f'{n:,}'.replace(',', '.')

# ═════════════════ TEMA DEEP SPACE ═══════════════════════════════════════════
_BG      = '#04060F'
_CARD    = '#0B1426'
_CARD2   = '#101F3C'
_BORDER  = '#15233E'
_FG      = '#D6E2F0'
_DIM     = '#8A99B5'
_BLUE    = '#58A6FF'
_CYAN    = '#22D3EE'
_GOLD    = '#FFC857'
_ORANGE  = '#FB923C'
_RED     = '#FF5C5C'

_F_TITLE = "font-family:'Orbitron','Courier New',monospace"
_F_MONO  = "font-family:'Share Tech Mono','Courier New',monospace"
_F_BODY  = "font-family:'Rajdhani','Segoe UI',sans-serif"

COLORES_DEF = {'stars': '#FFD08A', 'gas': '#4DA6FF'}

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': _BG,  'axes.facecolor' : _BG,
    'text.color'      : _FG,  'axes.labelcolor': _DIM,
    'xtick.color'     : _DIM, 'ytick.color'    : _DIM,
    'axes.edgecolor'  : _BORDER, 'savefig.facecolor': _BG,
    'font.family'     : 'monospace',
})


# Auto-escalado: el dashboard (diseño de ~2000px) se encoge para caber en
# cualquier ancho de ventana, manteniendo las proporciones.
_AJUSTE_JS = """
<script>
(function () {
  var DISENO = 2000;
  var previo = null;
  function ajustar() {
    // El scroller real de Voici/JupyterLab fija su alto en px via Lumino:
    // el zoom debe aplicarse ADENTRO (a su contenido), NUNCA al body ni al
    // scroller (escalar un alto fijado en px corta la pagina con negro).
    // El DOM aparece tarde (el kernel WASM duerme 2 s al arrancar), asi que
    // si el panel aun no existe no se escala nada y se reintenta.
    var cands = document.querySelectorAll('.lm-BoxPanel-child');
    var sc = null;
    for (var i = 0; i < cands.length; i++) {
      if (getComputedStyle(cands[i]).overflowY === 'auto') { sc = cands[i]; break; }
    }
    var objetivo = (sc && sc.firstElementChild) ? sc.firstElementChild : null;
    if (document.body.style.zoom) { document.body.style.zoom = ''; }
    if (previo && previo !== objetivo) {
      previo.style.zoom = '';
      previo.style.width = '';
      previo = null;
    }
    if (!objetivo) { return; }
    var z = Math.min(1, (document.documentElement.clientWidth || window.innerWidth) / DISENO);
    var zs = String(z);
    if (objetivo.style.width !== DISENO + 'px') { objetivo.style.width = DISENO + 'px'; }
    if (objetivo.style.zoom !== zs) { objetivo.style.zoom = zs; }
    previo = objetivo;
  }
  window.addEventListener('resize', function () { setTimeout(ajustar, 60); });
  ajustar();
  var n = 0;
  var timer = setInterval(function () {
    ajustar();
    if (++n >= 180) { clearInterval(timer); }
  }, 1000);
})();
</script>
"""

display(HTML(f'''
<style>
  @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700;900&family=Share+Tech+Mono&family=Rajdhani:wght@400;500;600;700&display=swap');
  /* ── Capa FIJA de cielo estrellado: tapa cualquier fondo claro del tema ── */
  html, body {{ background:{_BG} !important; }}
  .jp-Notebook, .jp-Cell, .jp-OutputArea-output, #notebook-container,
  #rendered_cells, .container, #notebook, #site, #page, .jp-NotebookPanel-notebook,
  .jp-WindowedPanel-outer, .jp-WindowedPanel-inner, main, #main
  {{ background:transparent !important; border:none !important; box-shadow:none !important; }}

  /* El contenido vive en z=1, sobre la capa de fondo */
  .jp-Cell-outputWrapper, .jp-OutputArea-output, .output_subarea
  {{ position:relative; z-index:1; }}

  #cielo-fondo {{
    position:fixed; top:0; left:0; right:0; bottom:0;
    z-index:0; pointer-events:none;
    background-color:{_BG};
    background-image:
      radial-gradient(1.6px 1.6px at 22px 34px,   rgba(255,255,255,0.90), transparent),
      radial-gradient(1px 1px     at 118px 88px,  rgba(190,225,255,0.80), transparent),
      radial-gradient(2.2px 2.2px at 168px 42px,  rgba(255,255,255,0.95), transparent),
      radial-gradient(1px 1px     at 64px 176px,  rgba(165,243,252,0.75), transparent),
      radial-gradient(1.4px 1.4px at 196px 150px, rgba(255,255,255,0.70), transparent),
      radial-gradient(1px 1px     at 30px 120px,  rgba(190,225,255,0.65), transparent),
      radial-gradient(1.8px 1.8px at 90px 300px,  rgba(255,255,255,0.85), transparent),
      radial-gradient(1px 1px     at 250px 200px, rgba(165,243,252,0.70), transparent),
      radial-gradient(1.3px 1.3px at 310px 90px,  rgba(255,255,255,0.65), transparent),
      radial-gradient(1px 1px     at 200px 320px, rgba(190,225,255,0.60), transparent),
      radial-gradient(2.4px 2.4px at 420px 240px, rgba(255,255,255,0.80), transparent),
      radial-gradient(1px 1px     at 480px 100px, rgba(165,243,252,0.65), transparent);
    background-size:
      230px 230px, 230px 230px, 230px 230px, 230px 230px, 230px 230px, 230px 230px,
      360px 360px, 360px 360px, 360px 360px,
      520px 520px, 520px 520px, 520px 520px;
    background-repeat: repeat;
    animation: derivaEstelar 360s linear infinite;
  }}
  @keyframes derivaEstelar {{
    from {{ background-position: 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0, 0 0; }}
    to   {{ background-position:
              -230px 230px, -230px 230px, -230px 230px, -230px 230px, -230px 230px, -230px 230px,
              -360px 360px, -360px 360px, -360px 360px,
              -520px 520px, -520px 520px, -520px 520px; }}
  }}
  .container {{ max-width:100% !important; width:99% !important; margin:0 auto !important; padding:0 !important; }}
  .jupyter-widgets {{ background:transparent !important; }}
  .jupyter-widgets, .widget-hbox, .widget-vbox,
  .jp-OutputArea-output, .jp-Cell-outputWrapper, .jp-OutputArea-child
  {{ overflow:visible !important; }}

  .widget-label   {{ color:{_DIM}  !important; {_F_MONO} !important; font-size:17px !important; }}
  .widget-readout {{ color:{_CYAN} !important; {_F_MONO} !important; font-size:17px !important; }}
  .widget-checkbox label {{ color:{_FG} !important; {_F_MONO} !important; font-size:17px !important; }}
  .widget-checkbox input {{ accent-color:{_CYAN}; transform:scale(1.4); margin-right:8px; }}
  select {{ background:{_CARD} !important; color:{_FG} !important;
            border:1px solid {_BORDER} !important; {_F_MONO} !important;
            font-size:17px !important; border-radius:6px !important; }}

  /* Sliders más gruesos y con glow */
  .ui-slider {{ background:{_BORDER} !important; border:none !important;
                height:7px !important; border-radius:4px !important; }}
  .ui-slider-handle {{ background:{_CYAN} !important; border:none !important;
                       width:17px !important; height:17px !important;
                       top:-5px !important; border-radius:50% !important;
                       box-shadow:0 0 9px rgba(34,211,238,0.7) !important; }}

  .jupyter-button {{ background:{_CARD} !important; color:{_DIM} !important;
                     border:1px solid {_BORDER} !important; {_F_MONO} !important;
                     font-size:17px !important; border-radius:8px !important;
                     letter-spacing:0.5px !important;
                     transition:all 0.18s ease !important; }}
  .jupyter-button:hover {{ color:{_CYAN} !important; border-color:{_CYAN} !important;
                           box-shadow:0 0 10px rgba(34,211,238,0.35) !important; }}
  .jupyter-button.mod-active {{ color:{_CYAN} !important; border-color:{_CYAN} !important;
                                background:#0E2238 !important; }}

  .p-Accordion .p-Collapse-header, .lm-Accordion .lm-Collapse-header,
  .jupyter-widget-Collapse-header {{ background:{_CARD} !important; color:{_FG} !important;
                     border-color:{_BORDER} !important; {_F_MONO} !important;
                     font-size:19px !important; padding:10px 16px !important; }}
  .p-Collapse-contents, .lm-Collapse-contents,
  .jupyter-widget-Collapse-contents {{ background:{_CARD} !important; border-color:{_BORDER} !important; }}
  .neon-box {{ border:1px solid rgba(34,211,238,0.65) !important;
               box-shadow:0 0 70px 8px rgba(34,211,238,0.10),
                          0 0 24px rgba(34,211,238,0.22),
                          0 6px 24px rgba(0,0,0,0.55) !important;
               border-radius:12px !important; }}
</style>
<div id="cielo-fondo"></div>
''' + _AJUSTE_JS))

In [ ]:
# ── CELDA 2: Interfaz interactiva ────────────────────────────────────────────
# En el navegador (WebAssembly), voici pierde mensajes comm_open si los widgets
# se crean durante el arranque del kernel ("removed socket"): esperar a que la
# conexión se estabilice evita widgets congelados. Sin efecto en voila local.
import sys, time, io
if sys.platform == 'emscripten':
    time.sleep(2.0)

def _fig_a_png(fig):
    """Renderiza la figura a PNG (bbox tight, como el renderer inline)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(),
                bbox_inches='tight')
    plt.close(fig)
    return buf.getvalue()
# Anchos: panel izq 470 + plot 940 + panel der 480 + 2 gaps de 18 = 1926 px
_PANEL_W  = '470px'
_RIGHT_W  = '480px'
_CENTER_W = '940px'
_DASH_W   = '1962px'

plt.rcParams['mathtext.fontset'] = 'stix'   # mathtext estilo LaTeX

# Fuentes para atributos style='...': las comillas internas DEBEN ser &quot;
# (una comilla simple literal cerraria el atributo y el navegador descartaria
#  todo lo que sigue: font-size, color, text-shadow...)
_F_TITLE = 'font-family:&quot;Orbitron&quot;,monospace'
_F_MONO  = 'font-family:&quot;Share Tech Mono&quot;,monospace'
_F_BODY  = 'font-family:&quot;Rajdhani&quot;,sans-serif'
_MATH    = ('font-family:&quot;STIX Two Math&quot;,&quot;Cambria Math&quot;,'
            '&quot;Times New Roman&quot;,serif;font-style:italic')

_W  = widgets.Layout(width='440px')
_ST = {'description_width': '200px'}

_DEFAULT_ALPHAS = {'gas': 0.30, 'stars': 0.55}

_CARD_STYLE = (f'background:linear-gradient(170deg,{_CARD},{_CARD2});'
               'border:1px solid rgba(34,211,238,0.65);border-radius:12px;'
               'box-shadow:0 0 70px 8px rgba(34,211,238,0.10),'
               ' 0 0 24px rgba(34,211,238,0.22),'
               ' 0 6px 24px rgba(0,0,0,0.55);')

# ═════════════════ BANNER PRINCIPAL (HTML estático, ancho completo) ═════════
# Se emite como HTML puro (no widget) para que ningún estilo de Voila lo atenúe.
_BANNER_STARS = (
    "radial-gradient(circle at 8% 30%, rgba(255,255,255,0.45) 1px, transparent 2.5px),"
    "radial-gradient(circle at 22% 75%, rgba(165,243,252,0.4) 1px, transparent 2.5px),"
    "radial-gradient(circle at 41% 20%, rgba(255,255,255,0.35) 1px, transparent 2.5px),"
    "radial-gradient(circle at 58% 65%, rgba(165,243,252,0.4) 1px, transparent 2.5px),"
    "radial-gradient(circle at 73% 35%, rgba(255,255,255,0.4) 1px, transparent 2.5px),"
    "radial-gradient(circle at 88% 70%, rgba(165,243,252,0.35) 1px, transparent 2.5px),"
    "radial-gradient(circle at 95% 25%, rgba(255,255,255,0.4) 1px, transparent 2.5px),"
)

display(HTML(
    f"<div style='background:{_BANNER_STARS}"
    f"linear-gradient(135deg,#0B1426 0%,#0E2238 55%,#101F3C 100%);"
    f"border:1px solid rgba(34,211,238,0.55);border-bottom:3px solid {_CYAN};border-radius:12px;"
    f"box-shadow:0 0 80px 10px rgba(34,211,238,0.12), 0 0 26px rgba(34,211,238,0.25), 0 6px 24px rgba(0,0,0,0.55), inset 0 0 90px rgba(34,211,238,0.07);"
    f"padding:30px 42px;width:{_DASH_W};box-sizing:border-box;margin:8px auto 4px;"
    f"opacity:1;position:relative;z-index:10;"
    f"display:flex;justify-content:space-between;align-items:flex-end;'>"
    f"<div>"
    f"<div style='{_F_TITLE};font-size:64px;font-weight:900;letter-spacing:13px;"
    f"color:#EAFDFF;"
    f"text-shadow:0 0 6px #FFFFFF, 0 0 14px #22D3EE, 0 0 36px #22D3EE,"
    f" 0 0 75px #22D3EE, 0 0 130px rgba(34,211,238,0.6);'>"
    f"◈ CIELO INTERACTIVO</div>"
    f"<div style='{_F_BODY};font-size:28px;font-weight:600;color:#DFF6FF;"
    f"letter-spacing:3px;margin-top:10px;"
    f"text-shadow:0 0 12px rgba(34,211,238,0.35);'>galaxias en tu computador"
    f"<span style='color:{_DIM};font-size:19px;font-weight:400;'>"
    f" &nbsp;·&nbsp; simulaciones cosmológicas del proyecto CIELO</span></div>"
    f"</div>"
    f"<div style='{_F_MONO};font-size:17px;color:{_DIM};text-align:right;"
    f"letter-spacing:1px;line-height:1.7;'>{sim_name} · Galaxia {gx_id}</div>"
    f"</div>"
))

# ═════════════════ PANEL DERECHO: propiedades + historia ═════════════════════
telemetry_html = widgets.HTML()

def _fval(v, fmt):
    return f'{v:{fmt}}' if (v is not None and np.isfinite(v)) else 'N/D'

def _math_lbl(txt_html):
    return f"<span style='{_MATH};font-size:21px;'>{txt_html}</span>"

def update_telemetry(snap):
    z, t_lb = snap_cache.get(snap, (None, None))
    mstar = mgas = lsfr = None
    if t_lb is not None and len(HIST_T):
        i = int(np.argmin(np.abs(HIST_T - (_T_UNIVERSE - t_lb))))
        mstar, mgas, lsfr = HIST_MS[i], HIST_MG[i], HIST_LSFR[i]

    if t_lb is None:
        t_str, z_str = 'N/D', 'N/D'
    elif t_lb < 0.05:
        t_str, z_str = 'hoy', f'{z:.4f}'
    else:
        t_str, z_str = f'{t_lb:.2f} Gyr atrás', f'{z:.4f}'

    filas = [
        ('Tiempo',                                     t_str,               _CYAN),
        (_math_lbl('z') + " <span style='font-size:14px;'>(corrimiento al rojo)</span>",
                                                       z_str,               _CYAN),
        ('', '', ''),
        (_math_lbl('log&#8201;M<sub>★</sub>') +
         " <span style='font-size:14px;'>(M<sub>☉</sub>)</span>",
                                                       _fval(mstar, '.2f'), _GOLD),
        (_math_lbl('log&#8201;M<sub>gas</sub>') +
         " <span style='font-size:14px;'>(M<sub>☉</sub>)</span>",
                                                       _fval(mgas,  '.2f'), '#60A5FA'),
        (_math_lbl('log&#8201;SFR') +
         " <span style='font-size:14px;'>(M<sub>☉</sub>/año)</span>",
                                                       _fval(lsfr,  '.2f'), _ORANGE),
    ]
    rows_html = []
    for k, v, vc in filas:
        if not k:
            rows_html.append(
                f"<div style='height:1px;background:linear-gradient(90deg,"
                f"{_BORDER},transparent);margin:11px 0;'></div>")
        else:
            rows_html.append(
                f"<div style='display:flex;justify-content:space-between;"
                f"align-items:baseline;line-height:2.2;'>"
                f"<span style='color:{_DIM}'>{k}</span>"
                f"<span style='color:{vc};font-size:20px;'>{v}</span></div>")
    telemetry_html.value = (
        f"<div style='{_CARD_STYLE}padding:18px 24px;{_F_MONO};font-size:19px;"
        f"width:{_RIGHT_W};box-sizing:border-box;'>"
        f"<div style='color:{_CYAN};{_F_TITLE};font-size:17px;"
        f"letter-spacing:1.5px;margin-bottom:2px;'>◈ PROPIEDADES GALAXIA CENTRAL</div>"
        f"<div style='height:2px;background:linear-gradient(90deg,{_CYAN}88,transparent);"
        f"margin:6px 0 9px;'></div>"
        f"<div style='color:{_DIM};font-size:14px;margin-bottom:10px;'>"
        f"1 Gyr = 1000 millones de años</div>"
        f"{''.join(rows_html)}</div>"
    )

# ── Historia de la galaxia ────────────────────────────────────────────────────
# Los widgets Output no se actualizan de forma confiable en voici (la captura
# por msg_id se pierde entre conexiones): las figuras van como Image (PNG por
# estado), el mismo mecanismo que usan las tarjetas HTML.
history_img = widgets.Image(format='png', layout=widgets.Layout(width='100%'))
history_out = widgets.VBox(
    [history_img],
    layout=widgets.Layout(width=_RIGHT_W, padding='10px 10px 6px'),
)
history_out.add_class('neon-box')

def draw_history(snap):
    z_s, t_lb_s = snap_cache.get(snap, (None, None))
    t_now = (_T_UNIVERSE - t_lb_s) if t_lb_s is not None else None
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(4.55, 5.5), dpi=100,
        facecolor=_CARD, sharex=True,
        gridspec_kw={'hspace': 0.14},
    )
    for ax in (ax1, ax2):
        ax.set_facecolor(_BG)
        ax.tick_params(labelsize=12, colors=_DIM)
        for sp in ax.spines.values():
            sp.set_edgecolor(_BORDER)

    m1 = np.isfinite(HIST_MS)
    if m1.any():
        t, ms = HIST_T[m1], HIST_MS[m1]
        ax1.fill_between(t, ms, ms.min() - 0.3, color=_GOLD, alpha=0.12)
        ax1.plot(t, ms, color=_GOLD, lw=2.5)
        ax1.set_ylim(ms.min() - 0.3, ms.max() + 0.35)
        if t_now is not None:
            i = np.argmin(np.abs(t - t_now))
            ax1.plot(t[i], ms[i], 'o', color=_RED, ms=8, markeredgewidth=0, zorder=6)
    m2 = np.isfinite(HIST_LSFR)
    if m2.any():
        tp, lsf = HIST_T[m2], HIST_LSFR[m2]
        ax2.fill_between(tp, lsf, lsf.min() - 0.3, color=_ORANGE, alpha=0.14)
        ax2.plot(tp, lsf, color=_ORANGE, lw=2.2)
        ax2.set_ylim(lsf.min() - 0.3, lsf.max() + 0.35)
        if t_now is not None:
            i = np.argmin(np.abs(tp - t_now))
            ax2.plot(tp[i], lsf[i], 'o', color=_RED, ms=8, markeredgewidth=0, zorder=6)

    if t_now is not None:
        for ax in (ax1, ax2):
            ax.axvline(t_now, color=_RED, lw=1.8, ls='--', alpha=0.9, zorder=5)

    ax1.set_xlim(0, _T_UNIVERSE + 0.4)
    ax1.set_ylabel(r'$\log\,M_{\star}\;[\mathrm{M}_{\odot}]$',
                   fontsize=15, color=_GOLD)
    ax2.set_ylabel(r'$\log\,\mathrm{SFR}\;[\mathrm{M}_{\odot}/\mathrm{a\tilde{n}o}]$',
                   fontsize=15, color=_ORANGE)
    ax2.set_xlabel('Edad del universo (Gyr)', fontsize=14, color=_DIM)
    ax1.set_title('HISTORIA DE LA GALAXIA', fontsize=15,
                  color=_CYAN, fontweight='bold', loc='left', pad=7)
    fig.tight_layout(pad=0.5)
    history_img.value = _fig_a_png(fig)

# ═════════════════ LÍNEA DE TIEMPO CÓSMICA ═══════════════════════════════════
timeline_html = widgets.HTML(layout=widgets.Layout(width=_CENTER_W, max_width=_CENTER_W))

_EPOCH_MARKS = [
    (0.05,  'Big Bang'),
    (0.4,   'Primeras galaxias'),
    (3.3,   'Mediodía cósmico'),
    (9.2,   'Nace el Sol'),
    (13.75, 'Hoy'),
]

def _cosmic_fact(edad):
    if edad < 1.0:
        return '🌅', ('Universo recién nacido: las primeras galaxias apenas se estaban '
                      'ensamblando a partir de nubes de gas primordial.')
    if edad < 3.0:
        return '💥', ('Época de crecimiento acelerado: las galaxias chocaban y se '
                      'fusionaban con frecuencia, ganando masa.')
    if edad < 6.0:
        return '✨', ('El "mediodía cósmico": el universo formaba estrellas al ritmo '
                      'más alto de toda su historia.')
    if edad < 8.9:
        return '🌌', ('El ritmo de formación estelar del universo ya descendía: '
                      'las galaxias entraban en su madurez.')
    if edad < 9.6:
        return '☀️', ('¡Por esta época se formaron el Sol y la Tierra, hace unos '
                      '4.600 millones de años!')
    if edad < 12.8:
        return '🌍', ('La Tierra ya existía: mientras esta galaxia evolucionaba, '
                      'la vida se desarrollaba en nuestro planeta.')
    if edad < 13.5:
        return '🪐', ('Pasado reciente: la galaxia ya tiene su aspecto adulto y '
                      'evoluciona de forma lenta y gradual.')
    return '🔭', ('Universo actual: así luce esta galaxia simulada hoy, tras '
                  '13.800 millones de años de evolución cósmica.')

def update_timeline(snap):
    z, t_lb = snap_cache.get(snap, (None, None))
    if t_lb is None:
        timeline_html.value = ''
        return
    edad = _T_UNIVERSE - t_lb
    pct  = max(0.0, min(100.0, edad / _T_UNIVERSE * 100))
    emoji, fact = _cosmic_fact(edad)

    marks = []
    for i, (t_m, label) in enumerate(_EPOCH_MARKS):
        p = t_m / _T_UNIVERSE * 100
        tx = '0' if p < 4 else ('-100%' if p > 96 else '-50%')
        arriba = (i % 2 == 1)
        tick_top = '21px' if arriba else '41px'
        lbl_top  = '0px'  if arriba else '57px'
        marks.append(
            f"<div style='position:absolute;left:{p:.1f}%;top:{tick_top};"
            f"width:1px;height:14px;background:{_DIM};opacity:0.5;'></div>"
            f"<div style='position:absolute;left:{p:.1f}%;top:{lbl_top};"
            f"transform:translateX({tx});color:{_DIM};font-size:15px;"
            f"letter-spacing:0.5px;white-space:nowrap;'>{label}</div>"
        )
    marks_html = ''.join(marks)

    timeline_html.value = (
        f"<div style='margin-top:26px;{_F_MONO};'>"
        f"<div style='color:{_CYAN};{_F_TITLE};font-size:16px;letter-spacing:2px;"
        f"margin:0 8px 2px;'>LÍNEA DE TIEMPO CÓSMICA</div>"
        f"<div style='height:2px;background:linear-gradient(90deg,{_CYAN}88,transparent);"
        f"margin:0 8px 5px;'></div>"
        f"<div style='position:relative;height:82px;margin:0 14px;'>"
        f"<div style='position:absolute;top:35px;left:0;right:0;height:9px;border-radius:5px;"
        f"background:linear-gradient(90deg,#3B0764,#1E3A8A 35%,#0E7490 70%,{_CYAN});"
        f"box-shadow:0 0 14px rgba(34,211,238,0.3);'></div>"
        f"{marks_html}"
        f"<div style='position:absolute;top:39px;left:{pct:.2f}%;"
        f"transform:translate(-50%,-50%);width:19px;height:19px;border-radius:50%;"
        f"background:{_RED};box-shadow:0 0 12px 4px rgba(255,92,92,0.7);"
        f"border:2px solid rgba(255,255,255,0.35);'></div>"
        f"</div>"
        f"<div style='{_CARD_STYLE}border-left:4px solid {_CYAN};"
        f"padding:15px 22px;margin:4px 8px 0;font-size:21px;color:{_FG};"
        f"line-height:1.5;{_F_BODY};font-weight:500;'>"
        f"<span style='font-size:24px;margin-right:11px;'>{emoji}</span>{fact}"
        f"</div>"
        f"<div style='text-align:right;color:{_DIM};opacity:0.55;font-size:13px;"
        f"margin:6px 10px 0;'>Proyecto CIELO · simulaciones cosmológicas "
        f"hidrodinámicas · prototipo CIELO Interactivo</div>"
        f"</div>"
    )

# ═════════════════ VIAJE EN EL TIEMPO ════════════════════════════════════════
time_card = widgets.HTML()

def _update_time_card(snap):
    z, t_lb = snap_cache.get(snap, (None, None))
    if t_lb is None:
        return
    edad = _T_UNIVERSE - t_lb
    if t_lb < 0.05:
        titulo  = 'HOY'
        detalle = f'universo actual · z = {z:.3f}'
    else:
        titulo  = f'hace {t_lb:.2f} Gyr'
        detalle = f'el universo tenía {edad:.1f} Gyr · z = {z:.3f}'
    time_card.value = (
        f"<div style='{_CARD_STYLE}padding:14px 16px;text-align:center;'>"
        f"<div style='color:{_CYAN};{_F_TITLE};font-size:34px;font-weight:700;"
        f"letter-spacing:1.5px;text-shadow:0 0 16px rgba(34,211,238,0.5);'>"
        f"{titulo}</div>"
        f"<div style='color:{_DIM};{_F_MONO};font-size:16px;margin-top:7px;"
        f"letter-spacing:0.5px;'>{detalle}</div></div>"
    )

time_idx_slider = widgets.IntSlider(
    min=0, max=len(_snaps_by_time) - 1,
    value=len(_snaps_by_time) - 1,
    description='', readout=False,
    layout=widgets.Layout(width='215px'),
    continuous_update=False,
    style={'description_width': '0px'},
)

play_btn = widgets.Play(
    min=0, max=len(_snaps_by_time) - 1,
    value=len(_snaps_by_time) - 1,
    interval=1100, step=1, show_repeat=False,
    layout=widgets.Layout(width='70px'),
)
speed_dd = widgets.Dropdown(
    options=[('Lento', 2000), ('Normal', 1100), ('Rápido', 500), ('Muy rápido', 300)],
    value=1100,
    layout=widgets.Layout(width='130px'),
)
widgets.jslink((play_btn, 'value'), (time_idx_slider, 'value'))
widgets.link((speed_dd, 'value'), (play_btn, 'interval'))

time_axis_lbl = widgets.HTML(
    f"<table style='width:100%;border-collapse:collapse;'><tr>"
    f"<td style='text-align:left;{_F_MONO};font-size:15px;color:{_DIM};'>◀ universo temprano</td>"
    f"<td style='text-align:right;{_F_MONO};font-size:15px;color:{_DIM};'>universo actual ▶</td>"
    f"</tr></table>",
    layout=widgets.Layout(width='100%'),
)

def _on_time_change(change):
    snap = _snaps_by_time[change.new]
    _update_time_card(snap)
    update_telemetry(snap)
    draw_history(snap)
    update_timeline(snap)

time_idx_slider.observe(_on_time_change, names='value')

_coord_names = {'centered': 'Centrada en la galaxia', 'rotated': 'Vista rotada'}
_coord_opts  = [(_coord_names.get(c, c), c) for c in META.get('coords', ['rotated'])]
coord_dd = widgets.Dropdown(
    options=_coord_opts,
    value='rotated' if any(v == 'rotated' for _, v in _coord_opts) else _coord_opts[0][1],
    description='Vista',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '65px'},
)

mode_tb = widgets.ToggleButtons(
    options=[('✦ Partículas', 'points'), ('☁ Difuso', 'diffuse')],
    value='points',
    style={'button_width': '212px'},
)

# ═════════════════ CONTROLES (solo 2 componentes) ════════════════════════════
alpha_w = {
    k: widgets.FloatSlider(value=_DEFAULT_ALPHAS[k], min=0, max=1, step=0.01,
                           description=NOMBRES[k],
                           layout=widgets.Layout(width='385px'),
                           style={'description_width': '120px'},
                           continuous_update=False)
    for k in ALL_KEYS
}
color_w = {
    k: widgets.ColorPicker(concise=True, value=COLORES_DEF[k],
                           layout=widgets.Layout(width='38px'))
    for k in ALL_KEYS
}
comp_rows = [
    widgets.HBox([color_w[k], alpha_w[k]],
                 layout=widgets.Layout(align_items='center', gap='8px'))
    for k in ['stars', 'gas']
]

size_sl  = widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1,
                               description='Tamaño partícula', layout=_W, style=_ST,
                               continuous_update=False)
xlim_sl  = widgets.FloatSlider(value=100, min=10, max=300, step=10,
                               description='Campo visual (kpc)', layout=_W, style=_ST,
                               continuous_update=False)
font_sl  = widgets.IntSlider(value=16, min=10, max=24, description='Tamaño fuente',
                             layout=_W, style=_ST, continuous_update=False)
data_dd  = widgets.Dropdown(
    options=[('Gas + Estrellas', 'both'), ('Solo estrellas', 'stars'), ('Solo gas', 'gas')],
    value='both', description='Mostrar',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '80px'})
glow_chk = widgets.Checkbox(value=True, description='Efecto brillo', indent=False,
                             layout=widgets.Layout(width='175px'))
full_chk = widgets.Checkbox(value=False, description='Resolución completa (más lento)',
                            indent=False, layout=widgets.Layout(width='330px'))

# ── Submuestreo de despliegue: cuota TOTAL proporcional estrellas:gas ─────────
_FAST_TOTAL = 160_000   # máx. partículas dibujadas en modo rápido

@lru_cache(maxsize=64)
def _fast_idx(snap, coord_type):
    comps = load_epoch(snap, coord_type)
    ns = len(comps.get('stars', _EMPTY)[0])
    ng = len(comps.get('gas', _EMPTY)[0])
    tot = ns + ng
    if tot <= _FAST_TOTAL:
        return None
    rng = np.random.default_rng(snap)          # semilla fija por época: imagen estable
    idx = {}
    for k, n in (('stars', ns), ('gas', ng)):
        if n:
            keep = min(n, max(1, int(round(_FAST_TOTAL * n / tot))))
            idx[k] = np.sort(rng.choice(n, keep, replace=False))
    return idx

def get_comps(snap, coord_type, full):
    """Partículas a dibujar: todas (full) o cuota proporcional reproducible."""
    comps = load_epoch(snap, coord_type)
    if full:
        return comps
    idx = _fast_idx(snap, coord_type)
    if idx is None:
        return comps
    return {k: (x[idx[k]], y[idx[k]]) for k, (x, y) in comps.items() if k in idx}

# ── Presets de exploración ───────────────────────────────────────────────────
_batch = {'on': False}
_preset_tick = widgets.IntText(value=0)   # disparador oculto: 1 redibujado por preset

PRESETS = {
    'Todo': dict(data='both', xlim=100, alphas=dict(_DEFAULT_ALPHAS)),
    'Estrellas': dict(data='stars', xlim=60, alphas={'stars': 0.70}),
    'Gas': dict(data='gas', xlim=120, alphas={'gas': 0.45}),
    'Entorno': dict(data='both', xlim=260, alphas={'stars': 0.50, 'gas': 0.35}),
}

def _apply_preset(cfg):
    _batch['on'] = True
    try:
        for k, v in cfg['alphas'].items():
            alpha_w[k].value = v
        data_dd.value = cfg['data']
        xlim_sl.value = cfg['xlim']
    finally:
        _batch['on'] = False
    _preset_tick.value += 1

preset_btns = []
for name, cfg in PRESETS.items():
    b = widgets.Button(description=name,
                       layout=widgets.Layout(width='102px', height='38px'))
    b.on_click(lambda _, c=cfg: _apply_preset(c))
    preset_btns.append(b)

# ── Guardar imagen (PNG de la vista actual) ──────────────────────────────────
save_btn    = widgets.Button(description='💾 Guardar imagen',
                             layout=widgets.Layout(width='215px', height='38px'))
save_status = widgets.HTML()
_last_args  = {}

def _on_save(_):
    if not _last_args:
        return
    fig  = render_galaxy_figure(**_last_args)
    snap = _snaps_by_time[_last_args['time_idx']]
    t_lb = snap_cache[snap][1]
    fn   = f'CIELO_{sim_name}_gx{gx_id}_hace{t_lb:.1f}Gyr.png'
    fig.savefig(fn, dpi=220, facecolor=_BG, bbox_inches='tight')
    plt.close(fig)
    save_status.value = (f"<span style='color:{_CYAN};{_F_MONO};font-size:14px;'>"
                         f"✓ guardado: {fn}</span>")

save_btn.on_click(_on_save)

# ── Guía educativa (fila completa, abierta por defecto) ──────────────────────
_GUIDE = [
    ('Estrellas', COLORES_DEF['stars'],
     'En las simulaciones, cada punto estelar es una partícula que representa '
     'a miles de estrellas nacidas juntas, con la misma edad y composición '
     'química. Las zonas más brillantes concentran más de ellas.'),
    ('Gas', COLORES_DEF['gas'],
     'El material difuso (principalmente hidrógeno y helio) del que nacen las '
     'estrellas. Cada partícula de gas lleva su propia densidad y temperatura: '
     'donde se enfría y comprime, se forman estrellas nuevas.'),
    ('Galaxias vecinas', _DIM,
     'Los grumos pequeños alrededor de la galaxia central son galaxias '
     'compañeras que pueden interactuar con ella y terminar fusionándose.'),
]
_guide_html = ''.join(
    f"<div style='margin-bottom:12px;break-inside:avoid;'>"
    f"<span style='display:inline-block;width:11px;height:11px;border-radius:50%;"
    f"background:{c};box-shadow:0 0 7px {c};margin-right:9px;'></span>"
    f"<span style='color:{c};font-weight:700;'>{n}:</span> "
    f"<span style='color:{_DIM};'>{d}</span></div>"
    for n, c, d in _GUIDE
)
guide_acc = widgets.Accordion(
    children=[widgets.HTML(
        f"<div style='{_F_BODY};font-size:20px;padding:14px 22px;line-height:1.55;'>"
        f"<div style='columns:3 360px;column-gap:48px;'>{_guide_html}</div>"
        f"<div style='color:{_DIM};{_F_MONO};font-size:15px;margin-top:11px;"
        f"border-top:1px solid {_BORDER};padding-top:10px;'>"
        f"Datos: simulaciones cosmológicas CIELO (Tissera et al. 2025). "
        f"La vista muestra la vecindad de la galaxia central "
        f"(una caja de ±250 kpc), no el volumen completo de la simulación.</div></div>"
    )],
    selected_index=0,
    layout=widgets.Layout(width=_DASH_W, max_width=_DASH_W),
)
guide_acc.set_title(0, '📖 ¿Qué estoy viendo?')
guide_acc.add_class('neon-box')

# ═════════════════ RENDER ════════════════════════════════════════════════════
def _scale_bar(ax, xlim, font_size):
    L  = min([5, 10, 20, 25, 50, 100], key=lambda c: abs(c - xlim / 3.5))
    x0 = -0.93 * xlim
    y0 = -0.92 * xlim
    ax.plot([x0, x0 + L], [y0, y0], color=_CYAN, lw=2.4, solid_capstyle='butt', alpha=0.85)
    ly = _fmt_n(int(L * 3262))
    ax.text(x0, y0 + 0.022 * xlim, f'{L} kpc ≈ {ly} años luz',
            color=_DIM, fontsize=max(font_size - 2, 9), fontfamily='monospace')

def _render_density(ax, comps, colors, alphas, visibles, xlim):
    """Mapa difuso por composición aditiva de capas de densidad."""
    bins = 520
    img  = np.zeros((bins, bins, 3))
    rng  = [[-xlim, xlim], [-xlim, xlim]]
    for k in visibles:
        x, y = comps.get(k, _EMPTY)
        a = alphas[k]
        if a <= 0 or len(x) == 0:
            continue
        H, _, _ = np.histogram2d(x, y, bins=bins, range=rng)
        H = H.T
        if _HAS_SCIPY:
            H = gaussian_filter(H, 1.3)
        if H.max() <= 0:
            continue
        pos = H[H > 0]
        H = np.arcsinh(H / (np.percentile(pos, 55) + 1e-9))
        H /= H.max()
        rgb  = np.asarray(mcolors.to_rgb(colors[k]))
        img += H[..., None] * rgb * (a * 1.6)
    img = 1.0 - np.exp(-1.6 * img)
    ax.imshow(np.clip(img, 0, 1), extent=[-xlim, xlim, -xlim, xlim],
              origin='lower', interpolation='bilinear', zorder=1)

def render_galaxy_figure(
    time_idx, coord_type, data_type, mode,
    alphas, colors, pt_size, font_size, xlim, glow, full_res,
):
    snap    = _snaps_by_time[time_idx]
    # el modo difuso usa siempre los datos completos (el costo no depende de N)
    comps   = get_comps(snap, coord_type, full_res or mode == 'diffuse')
    z, t_lb = snap_cache.get(snap, (None, None))

    if data_type == 'stars':
        visibles = ['stars']
    elif data_type == 'gas':
        visibles = ['gas']
    else:
        visibles = ['gas', 'stars']   # estrellas encima del gas

    fig, ax = plt.subplots(figsize=(9.4, 9.4), dpi=100, facecolor=_BG)
    ax.set_facecolor(_BG)

    if not comps:
        ax.text(0.5, 0.5, 'Sin datos para esta época',
                transform=ax.transAxes, ha='center', color=_DIM,
                fontsize=font_size + 2, fontfamily='monospace')
    elif mode == 'diffuse':
        _render_density(ax, comps, colors, alphas, visibles, xlim)
    else:
        for k in visibles:
            x, y = comps.get(k, _EMPTY)
            a = alphas[k]
            if a <= 0 or len(x) == 0:
                continue
            if glow:
                ax.scatter(x, y, s=pt_size * 10, color=colors[k],
                           alpha=min(a * 0.10, 1.0), edgecolors='none', rasterized=True)
            ax.scatter(x, y, s=pt_size, color=colors[k], alpha=a,
                       edgecolors='none', rasterized=True)

    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-xlim, xlim)
    ax.set_aspect('equal')

    if t_lb is not None and t_lb < 0.05:
        t_str = 'hoy'
    elif t_lb is not None:
        t_str = f'hace {t_lb:.2f} Gyr'
    else:
        t_str = ''
    z_str = f'z = {z:.3f}' if z is not None else ''
    ax.text(0.02, 0.98, f'CIELO · {sim_name} · Galaxia {gx_id}',
            transform=ax.transAxes, color='white',
            fontsize=font_size + 2, fontweight='bold', va='top', fontfamily='monospace')
    ax.text(0.02, 0.945, f'{t_str}   {z_str}',
            transform=ax.transAxes, color=_CYAN,
            fontsize=font_size + 1, va='top', fontfamily='monospace')

    _scale_bar(ax, xlim, font_size)

    ax.set_xlabel(r'$x$ [kpc]', fontsize=font_size, color=_DIM)
    ax.set_ylabel(r'$y$ [kpc]', fontsize=font_size, color=_DIM)
    ax.tick_params(labelsize=font_size - 1, colors=_DIM)
    for sp in ax.spines.values():
        sp.set_edgecolor(_BORDER)

    handles = [
        Line2D([0], [0], marker='o', markersize=9, color='none',
               markerfacecolor=colors[k], label=NOMBRES[k])
        for k in visibles if alphas[k] > 0.01 and len(comps.get(k, _EMPTY)[0])
    ]
    if handles:
        ax.legend(handles=handles, ncol=2, fontsize=max(font_size, 13),
                  loc='lower right', facecolor=_CARD, edgecolor=_BORDER,
                  labelcolor=_FG, framealpha=0.92)
    fig.tight_layout()
    return fig

def plot_galaxy(
    time_idx, coord_type, data_type, mode,
    a_stars, a_gas, c_stars, c_gas,
    pt_size, font_size, xlim, glow, full_res, tick,
):
    if _batch['on']:
        return
    alphas = {'stars': a_stars, 'gas': a_gas}
    colors = {'stars': c_stars, 'gas': c_gas}
    _last_args.clear()
    _last_args.update(dict(
        time_idx=time_idx, coord_type=coord_type, data_type=data_type,
        mode=mode, alphas=alphas, colors=colors,
        pt_size=pt_size, font_size=font_size, xlim=xlim, glow=glow,
        full_res=full_res,
    ))
    fig = render_galaxy_figure(**_last_args)
    galaxy_img.value = _fig_a_png(fig)

galaxy_img = widgets.Image(format='png', layout=widgets.Layout(width='100%'))
out = widgets.VBox(
    [galaxy_img],
    layout=widgets.Layout(width=_CENTER_W, max_width=_CENTER_W, overflow='hidden'),
)

_PLOT_FUENTES = {
    'time_idx': time_idx_slider, 'coord_type': coord_dd,
    'data_type': data_dd, 'mode': mode_tb,
    'a_stars': alpha_w['stars'], 'a_gas': alpha_w['gas'],
    'c_stars': color_w['stars'], 'c_gas': color_w['gas'],
    'pt_size': size_sl, 'font_size': font_sl, 'xlim': xlim_sl,
    'glow': glow_chk, 'full_res': full_chk,
    'tick': _preset_tick,
}

def _redibuja(*_):
    plot_galaxy(**{k: w.value for k, w in _PLOT_FUENTES.items()})

for _w_src in _PLOT_FUENTES.values():
    _w_src.observe(_redibuja, 'value')
_redibuja()

# Estado inicial de paneles laterales
_snap0 = _snaps_by_time[-1]
_update_time_card(_snap0)
update_telemetry(_snap0)
draw_history(_snap0)
update_timeline(_snap0)


# ═════════════════ LAYOUT FINAL ══════════════════════════════════════════════
def _hdr(txt):
    return widgets.HTML(
        f"<div style='color:{_CYAN};{_F_MONO};font-size:17px;"
        f"letter-spacing:2.5px;padding:13px 12px 3px;'>{txt}</div>"
        f"<div style='height:2px;background:linear-gradient(90deg,{_CYAN}55,transparent);"
        f"margin:0 12px 6px;'></div>"
    )

ctrl_panel = widgets.VBox([
    _hdr('⏳ VIAJE EN EL TIEMPO'),
    widgets.VBox([
        time_card,
        widgets.HBox([play_btn, speed_dd, time_idx_slider],
                     layout=widgets.Layout(align_items='center', gap='8px',
                                           justify_content='center')),
        time_axis_lbl,
    ], layout=widgets.Layout(padding='4px 12px 8px')),
    _hdr('🎛 EXPLORACIÓN RÁPIDA'),
    widgets.HBox(preset_btns, layout=widgets.Layout(padding='0 12px 8px', gap='8px')),
    _hdr('👁 MODO DE VISUALIZACIÓN'),
    widgets.VBox([mode_tb], layout=widgets.Layout(padding='0 12px 8px')),
    _hdr('🎨 COMPONENTES'),
    widgets.VBox(comp_rows, layout=widgets.Layout(padding='0 12px 8px', gap='7px')),
    _hdr('⚙ AJUSTES DE LA VISTA'),
    widgets.VBox([
        size_sl, xlim_sl, font_sl,
        widgets.HBox([data_dd, glow_chk],
                     layout=widgets.Layout(align_items='center', gap='10px')),
        coord_dd,
        full_chk,
        widgets.HBox([save_btn], layout=widgets.Layout(padding='6px 0 0 0')),
        save_status,
    ], layout=widgets.Layout(padding='0 12px 14px', gap='5px')),
], layout=widgets.Layout(width=_PANEL_W, min_width=_PANEL_W))
ctrl_panel.add_class('neon-box')

center_panel = widgets.VBox(
    [out, timeline_html],
    layout=widgets.Layout(width=_CENTER_W, max_width=_CENTER_W, overflow='hidden'),
)

right_panel = widgets.VBox(
    [telemetry_html, history_out],
    layout=widgets.Layout(gap='30px', width=_RIGHT_W, min_width=_RIGHT_W),
)

dashboard_row = widgets.HBox(
    [ctrl_panel, center_panel, right_panel],
    layout=widgets.Layout(gap='36px', align_items='flex-start'),
)

# Todo el conjunto centrado: el espacio sobrante se reparte a ambos lados
display(widgets.VBox(
    [dashboard_row, guide_acc],
    layout=widgets.Layout(gap='34px', align_items='center', width='100%',
                          padding='6px 26px 26px'),
))